# Shazam (Part 2)

A Running exercise to replicate the Shazam algorithm.

## Goals

- More practice with pandas `DataFrames`
- Building intuition for distances
- Experimenting with evaluation strategies

## Setup

Run the following 2 cells to import all necessary files, libraries and helpers for this exercise.

In [ ]:
!mkdir -p data/audio
!wget -q https://github.com/PSAM-5020-2026F-A/5020-utils/raw/refs/heads/main/datasets/audio/hood-internet.tar.gz
!tar -xzf ./hood-internet.tar.gz -C ./data/audio/ && rm -rf hood-internet.tar.gz*

In [ ]:
import librosa
import matplotlib.pyplot as plt
import pandas as pd
import random

from IPython.display import Audio
from os import listdir

### ❗️❗️ WARNING ❗️❗️

The songs we'll be using are from mixtapes by [The Hood Internet](https://www.thehoodinternet.com/). They were part of a wave of _mashup_ producers in the early $2\text{,}000\text{s}$ that mixed _classic_ american indie, pop and rap music from other time periods.

We're using their mixtapes here mostly because: they're available freely on the internet, they encompass a good variety of songs and styles, and there are a good number of songs; not too few nor too many.

The lyrics mostly come from the rap songs being sampled and contain references to all the things that rap artists were rapping about in the $90\text{s}$ and $00\text{s}$ (sex, drugs, guns, money...).

We definitely don't have to listen to the songs to complete the exercise.

We can consider alternative datasets for this exercise. Feel free to adapt the exercise to use files from different albums, just indicate that here in the notebook.

## Shazam !

Ok. Let's continue our journey of replicating the Shazam algorithm.

As a review: it's the same type of classification task we've been looking at, but instead of classifying a piece of audio into one of $3$ classes, the algorithm has to pick which of the millions of songs the piece of audio most likely came from.

Instead of manually extracting features, we need something a little more systematic.

Mel-Frequency Cepstral Coefficients (MFCC) is a techninque for automatically extracting meaningful numeric information from audio, independent of the type of audio.

We'll continue to use the librosa library to help us extract MFCCs from audio.

### Review: MFCCs

In [ ]:
# First, let's grab all audio file names from our album directory

TRACKS_DIR = "./data/audio/hood-internet/mixtape04"
fnames = sorted([f for f in listdir(TRACKS_DIR) if f.endswith("mp3")])

In [ ]:
# opening up a random song (index 4)
y,sr = librosa.load(f"{TRACKS_DIR}/{fnames[4]}")

plt.plot(y)
plt.show()

display(Audio(y, rate=sr))

In [ ]:
# Extracting the mfccs for track at index 4.
# We use mean(axis=1) to get a single value for each frequency channel, independent of time.

mfccs = librosa.feature.mfcc(y=y, n_mfcc=10).mean(axis=1)

print(mfccs.shape)
display(mfccs)

### Review: Extract MFCC features from all songs

It takes about $20$ seconds to extract features for all songs.

In [ ]:
# Helper function
def get_features(y, n_mfcc=20):
  mfeats = {}
  mfccs = librosa.feature.mfcc(y=y, n_mfcc=n_mfcc).mean(axis=1)
  for idx,fcc in enumerate(mfccs):
    mfeats[f"mfcc{idx}"] = float(fcc)
  return mfeats

In [ ]:
# This cell takes about 20-30 seconds

track_feats = []

for f in fnames:
  fpath = f"{TRACKS_DIR}/{f}"
  y, sr = librosa.load(fpath)
  track_feats.append(get_features(y=y))

tracks_df = pd.DataFrame.from_records(track_feats)
tracks_df

### Review: Shazam

This is what's behind the [Shazam algorithm](https://www.youtube.com/watch?v=b6xeOLjeKs0): a large database of songs, where each song is described with only a few features so they can be organized, searched, queried and compared very very quickly.

Another [video](https://www.youtube.com/watch?v=kMNSAhsyiDg) reference.

Let's implement something like this.

We're focusing on a single album, with about $30$ songs, but the same strategy (with minor modifications) will work for datasets of many more songs.

Here's the outline of what we are doing to do:

1. Extract MFCC features from all songs in our database (single album) ✅
2. Create a dataset of song features ✅
3. Given an input sound clip: extract its MFCC features
4. Find closest song in database by feature distance

### Multi-dimensional distance

We stopped with this challenge last time.

The next step is to create a function that, given a set of mfcc features, finds the song in our dataset with the most similar features.

This is similar to one of the exercises in Homework02, where we found the closest cities to a given city; and similar to the MoMA exercise, where we found the closest apartment to a location.

The difference is that in those cases our distance was calculated using $2$ features (latitude and longitude), but now our distances will be calculated using $20$ (or so) features.

The math is the same. Since the number of features is not fixed and is greater than $5$, we should use a loop, possibly with [`row.index`](https://pandas.pydata.org/docs/reference/api/pandas.Series.index.html) (similar to `df.columns`) or `zip()`...

In [ ]:
# TODO: implement a function that returns distances between 2 mfcc feature lists
def mfcc_dist(mfccA, mfccB):
  sum_of_squares = 0
  # TODO: iterate over pairs of features
  #       calculate the sum of the square of their differences
  #       return the square root of that sum
  return sum_of_squares ** 0.5

### 3. Given an input sound clip: extract its MFCC features

In [ ]:
# Let's open the track at index 4 and get 5 seconds of audio from the center of the song

# Track at index 4
fpath = f"{TRACKS_DIR}/{fnames[4]}"
y_in, sr_in = librosa.load(fpath)

# get 5 seconds from middle of song
midx = int(len(y_in) // 2)
half_win = int(2.5 * sr_in)
samples_5sec = y_in[midx - half_win : midx + half_win]

feat_in = get_features(samples_5sec)

# And create DataFrame with MFCC features for the 5-second clip
# A Series is like a 1-row DataFrame, which is what we have for the MFCCs of a single song clip

in_df = pd.Series(feat_in)
in_df

### 4. Find closest song in database by feature distance

We have MFCCs for all songs and MFCCs for the wanted clip.

We'll go through the dataset and create a new feature called `distance` that holds the MFCC distance between all songs and the target.

We can loop through a `DataFrame` using [`df.iterrows()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html), or we can apply a function to all rows using [`df.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html).

In [ ]:
# TODO: loop through rows or apply a function to all rows
# TODO: add results to song DataFrame as a column called "distance"

### Almost there

We have the distance values in a column, so we just have to find the index of the row with the smallest distance.

There are many ways of doing this, but a general strategy that can be easily extended later when evaluating our algorithm, is to use [`sort_values()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html), and [`index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.index.html).

Using [`idxmin()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.idxmin.html) is easier, but it only gives us the closest row index.

When the prediction is wrong, using the [`sort_values()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) strategy will also tell us how far the correct index was from being predicted.

In [ ]:
# TODO: find index of row with smallest "distance" value
# TODO: sort DataFrame by distance values
# TODO: look at first index

### 5. Evaluate

We should calculate how often our Shazam algorithm is right.

This can be done by implementing the following set of steps:
1. Initialize a counter to keep track of number of correct predictions
2. For $N$ predictions (where $N$ can be $10$, $100$, $1000$, etc ...)
3. Pick a random track from the album
4. Pick a random $5$ second clip from the random track
5. Extract features, compare to features in `tracks_df` and predict which track it is
6. If prediction correct, increment correct predictions counter

Alternatively, instead of only incrementing the correct predictions counter when the correct track is the very first on the list of possible tracks, we can increment the counter if the correct track is in the $\text{top-}3$ predictions. This is the $\text{top-}3$ accuracy of our classifier, which, instead of being a binary measurement of right/wrong, also gives us a bit of credit for being "close".

In [ ]:
# TODO: write a function called random5() that
#       receives an audio file path, extracts MFCC features from 
#       a random 5-second portion of the audio and returns a Series with the features

def random5(fpath):
  # TODO: possible strategy:
  #       load song
  #       convert 5 seconds to number of samples
  #       random.randrange() can help pick the starting sample between 0 and 5 seconds from the end of the song.
  #       use slicing to extract the samples corresponding to the selected 5-second range
  #       run get_features() and convert to Series before returning

  return pd.Series([])

We'll also need a function for comparing the `DataFrame` of songs and the `DataFrame` of the input clip.

We just have to wrap the distance calculation and the search for the smallest distance from above in a function.


Again, we should use [`sort_values()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) here and return a list of the predictions. This way when the first prediction is wrong we can see how wrong it was by finding the correct answer in the list of predictions.

In [ ]:
# TODO: implement comparison function
def compare_predict(tracks_df, in_df):
  # TODO: repeat code from above to find index of row with the shortest distance
  return 0

Now we use the above defined functions to run some tests.

With the `random5()` and `compare_predict()` functions defined above we should be able to just run the following cell to evaluate our classifier.

We want to eventually run $200$ to $500$ tests, but we can start with $10$ just to make sure the code works.

In [ ]:
num_tests = 10
num_correct = 0
num_correct_top3 = 0

for _ in range(num_tests):
  # pick a random song
  rtrack_idx = random.randrange(0, len(fnames))
  rtrack_name = fnames[rtrack_idx]

  # extract features from random 5-second clip of song
  in_df = random5(f"{TRACKS_DIR}/{rtrack_name}")

  # compare, and keep track of correct predictions
  predictions = compare_predict(tracks_df, in_df)

  if rtrack_idx == predictions[0]:
    num_correct += 1
  if rtrack_idx in predictions[:3]:
    num_correct_top3 += 1

# print accuracy (number of correct predictions divided by total tests)
print(f"top 1: {round(num_correct / num_tests * 100, 2)}%")
print(f"top 3: {round(num_correct_top3 / num_tests * 100, 2)}%")

### Improve the Algorithm ?

Not bad.

(Re)Watch the Shazam video [here](https://www.youtube.com/watch?v=b6xeOLjeKs0). What's the one big difference between their algorithm and ours ?

(You don't have to improve the code. Just think about what could be some potential next steps.)